In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    to_timestamp,
    current_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
BRONZE_TABLE = "weather_streaming.weather.weather_bronze"

SILVER_TABLE = "weather_streaming.weather.weather_silver"

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

In [0]:
silver_df = bronze_df.withColumn("city", trim(col("city")))
silver_df = silver_df.filter(col("latitude").between(-90, 90)).filter(
    col("longitude").between(-180, 180)
)

In [0]:
silver_df = (
    silver_df.filter(col("humidity_pct").between(0, 100) | col("humidity_pct").isNull())
    .filter(col("wind_speed_kmh") >= 0)
    .filter(col("precipitation_mm") >= 0)
)
silver_df = silver_df.withColumn(
    "temperature_category",
    when(col("temperature_c") < 15, "COLD")
    .when(col("temperature_c") < 25, "MODERATE")
    .when(col("temperature_c") < 35, "WARM")
    .otherwise("HOT"),
)
silver_df = silver_df.dropDuplicates(["city", "event_time"])

In [0]:
dedup_window = Window.partitionBy("city", "event_time").orderBy(
    col("ingestion_time").desc()
)

silver_df = (
    silver_df.withColumn("rn", row_number().over(dedup_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
silver_df = silver_df.select(
    "city",
    "latitude",
    "longitude",
    "event_time",
    "temperature_c",
    "humidity_pct",
    "wind_speed_kmh",
    "precipitation_mm",
    "ingestion_time",
    "source",
    "pipeline_run_id",
    "source_file",
    "temperature_category",
    "bronze_processed_at",
    "record_hash",
    "year",
    "month",
    "day",
)

In [0]:
silver_df.createOrReplaceTempView("weather_silver_source")

In [0]:
spark.sql(
    f"""
    MERGE INTO {SILVER_TABLE} AS target

    USING weather_silver_source AS source

    ON
        target.city = source.city
        AND target.event_time = source.event_time

    WHEN MATCHED
        AND source.ingestion_time > target.ingestion_time
    THEN UPDATE SET
        target.latitude = source.latitude,
        target.longitude = source.longitude,

        target.temperature_c = source.temperature_c,
        target.humidity_pct = source.humidity_pct,
        target.wind_speed_kmh = source.wind_speed_kmh,
        target.precipitation_mm = source.precipitation_mm,

        target.ingestion_time = source.ingestion_time,

        target.source = source.source,
        target.pipeline_run_id = source.pipeline_run_id,
        target.source_file = source.source_file,

        target.temperature_category =
            source.temperature_category,

        target.bronze_processed_at =
            source.bronze_processed_at,

        target.record_hash =
            source.record_hash,

        target.year = source.year,
        target.month = source.month,
        target.day = source.day

    WHEN NOT MATCHED THEN INSERT
    (
        city,
        latitude,
        longitude,
        event_time,

        temperature_c,
        humidity_pct,
        wind_speed_kmh,
        precipitation_mm,

        ingestion_time,

        source,
        pipeline_run_id,
        source_file,

        temperature_category,

        bronze_processed_at,
        record_hash,

        year,
        month,
        day
    )

    VALUES
    (
        source.city,
        source.latitude,
        source.longitude,
        source.event_time,

        source.temperature_c,
        source.humidity_pct,
        source.wind_speed_kmh,
        source.precipitation_mm,

        source.ingestion_time,

        source.source,
        source.pipeline_run_id,
        source.source_file,

        source.temperature_category,

        source.bronze_processed_at,
        source.record_hash,

        source.year,
        source.month,
        source.day
    )
    """
)

In [0]:
silver_count = silver_df.count()

print("=" * 60)
print("BRONZE → SILVER COMPLETED")
print("=" * 60)
print(f"Records prepared : {silver_count}")
print(f"Target table     : {SILVER_TABLE}")
print("=" * 60)